In [0]:
# =============================================================================
# NB11 — Feature-group ablation with repeated grouped CV
# =============================================================================
# Objective:
#   NB10 v2 gave the first positive grouped-CV R² (XGBoost L3 = 0.153 ± 0.133).
#   Before interpreting it, answer two questions:
#     Q1. Do material descriptors add anything once experimental conditions
#         are known?  → compare L3 vs EXP_ONLY (X_exp_v2 without X_material).
#     Q2. Which experimental group carries the gain? In particular, is it the
#         per-gram normalisation artefact (catalyst mass)?  → L3 minus each
#         group ("drop-one-group" ablation).
#   And quantify uncertainty: repeated GroupKFold by DOI with N_REPEATS
#   different random assignments of DOIs to folds. ΔR² vs L3 is computed
#   PAIRED on identical splits (much tighter than comparing two means).
#
#   Note: NB07's main_cv splits remain the reference protocol; the repeated
#   splits here are an additional uncertainty estimate, still grouped by DOI
#   (no paper ever in train and val simultaneously).
#
# Model: XGBoost, full feature set (best in NB10 v2). Same hyperparameters.
# Inputs:  master, X_material, X_thermo, X_exp_v2
# Outputs: ml_perovskites.ablation_nb11  (repeat × fold × config)
#          /tmp/nb11_ablation.png
# Rules:   target encoding + imputation inside each fold; SEED = 42
# =============================================================================

import warnings
warnings.filterwarnings("ignore")

import subprocess, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.impute  import SimpleImputer
from sklearn.metrics import r2_score
from pyspark.sql import SparkSession

SEED      = 42
SCHEMA    = "ml_perovskites"
N_FOLDS   = 5
N_REPEATS = 5
TE_M      = 10

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "8")

try:
    import xgboost as xgb
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "xgboost"])
    import xgboost as xgb

# ---------------------------------------------------------------------------
# 1. Load and align (same conventions as NB10)
# ---------------------------------------------------------------------------
master    = spark.table(f"{SCHEMA}.master").toPandas()
X_mat_raw = spark.table(f"{SCHEMA}.X_material").toPandas()
X_thm_raw = spark.table(f"{SCHEMA}.X_thermo").toPandas()
X_exp_raw = spark.table(f"{SCHEMA}.X_exp_v2").toPandas()

def is_use_h2(s):
    return s.astype(str).str.strip().str.lower().isin(["true", "1"])

master["y_log10_1p_final"] = pd.to_numeric(master["y_log10_1p_final"], errors="coerce")
df_base = master[is_use_h2(master["use_h2"]) &
                 master["y_log10_1p_final"].notna() &
                 master["doi_key"].notna()].copy()
df_base["y"] = df_base["y_log10_1p_final"].astype(float)

for t in [df_base, X_mat_raw, X_thm_raw, X_exp_raw]:
    t["source_row_id"] = t["source_row_id"].astype(str).str.strip()

META = {"source_row_id", "base_compound", "family_proposed", "formula_base_paper",
        "A_site_paper", "B_site_paper", "doi", "doi_key", "y_log10_1p_final",
        "y_log10_1p", "h2_rate_umol_g_h", "h2_umol_h_rate_mass"}

for raw in [X_mat_raw, X_thm_raw]:
    for c in raw.columns:
        if c != "source_row_id":
            raw[c] = pd.to_numeric(raw[c], errors="coerce")

CAT_COLS = [c for c in ["co_catalyst_type", "run_type", "scavenger_class",
                        "lamp_type", "prep_class"] if c in X_exp_raw.columns]

x_mat = [c for c in X_mat_raw.columns if c not in META
         and pd.api.types.is_numeric_dtype(X_mat_raw[c])
         and X_mat_raw[c].isna().mean() < 0.50]
x_thm = [c for c in X_thm_raw.columns if c not in META
         and pd.api.types.is_numeric_dtype(X_thm_raw[c])
         and X_thm_raw[c].isna().mean() < 0.99]
x_exp_num = [c for c in X_exp_raw.columns if c not in ["source_row_id"] + CAT_COLS]
for c in x_exp_num:
    X_exp_raw[c] = pd.to_numeric(X_exp_raw[c], errors="coerce")
x_exp = x_exp_num + CAT_COLS

df = (df_base[["source_row_id", "doi_key", "y"]]
      .merge(X_mat_raw[["source_row_id"] + x_mat], on="source_row_id", how="left")
      .merge(X_thm_raw[["source_row_id"] + x_thm], on="source_row_id", how="left")
      .merge(X_exp_raw[["source_row_id"] + x_exp], on="source_row_id", how="left"))
print(f"Merged: {df.shape} | material {len(x_mat)} | thermo {len(x_thm)} | exp {len(x_exp)}")

# ---------------------------------------------------------------------------
# 2. Experimental feature groups (physical grouping)
# ---------------------------------------------------------------------------
GROUPS = {
    "mass_normalisation": ["catalyst_mass_g", "log10_catalyst_mass_g", "catalyst_conc_g_L"],
    "reactor_light":      ["lamp_power_W", "lamp_power_per_mL", "cutoff_nm_final",
                           "lamp_type", "volume_mL"],
    "solution_reaction":  ["organic_vol_fraction", "T_reaction_C", "reaction_time_h",
                           "scavenger_class", "run_type", "substrate_is_target"],
    "cocatalyst":         ["co_catalyst_type", "co_catalyst_loading"],
    "measured_material":  ["particle_size_nm", "ssa_m2_g"],
    "synthesis":          ["prep_class", "prep_multistep", "thermal_T_C",
                           "thermal_time_h", "thermal_T_le_250C"],
}
GROUPS = {g: [c for c in cols if c in x_exp] for g, cols in GROUPS.items()}
ungrouped = [c for c in x_exp if not any(c in v for v in GROUPS.values())]
print("Groups:", {g: len(v) for g, v in GROUPS.items()}, "| ungrouped:", ungrouped)

L3 = x_mat + x_thm + x_exp
CONFIGS = {
    "L1_material_only": x_mat,
    "EXP_ONLY":         x_exp,
    "L3_full":          L3,
}
for g, cols in GROUPS.items():
    CONFIGS[f"L3_minus_{g}"] = [c for c in L3 if c not in cols]

# ---------------------------------------------------------------------------
# 3. Repeated grouped splits (random DOI → fold assignment per repeat)
# ---------------------------------------------------------------------------
dois = np.array(sorted(df["doi_key"].unique()))

def grouped_splits(repeat):
    rng = np.random.default_rng(SEED + repeat)
    perm = rng.permutation(dois)
    fold_of = {d: i % N_FOLDS for i, d in enumerate(perm)}
    f = df["doi_key"].map(fold_of).values
    return [(np.where(f != k)[0], np.where(f == k)[0]) for k in range(N_FOLDS)]

def encode(tr, va, cols):
    Xtr, Xva = tr[cols].copy(), va[cols].copy()
    for cc in [c for c in CAT_COLS if c in cols]:
        gm = tr["y"].mean()
        st = tr.groupby(cc)["y"].agg(["mean", "count"])
        enc = ((st["mean"] * st["count"] + gm * TE_M) / (st["count"] + TE_M)).to_dict()
        Xtr[cc] = tr[cc].map(enc).fillna(gm)
        Xva[cc] = va[cc].map(enc).fillna(gm)
    return Xtr.astype(float), Xva.astype(float)

def model():
    return xgb.XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=4,
                            subsample=0.8, colsample_bytree=0.7, min_child_weight=3,
                            reg_lambda=1.0, tree_method="hist",
                            random_state=SEED, verbosity=0)

rows = []
for rep in range(N_REPEATS):
    splits = grouped_splits(rep)
    for k, (tr_i, va_i) in enumerate(splits):
        tr, va = df.iloc[tr_i], df.iloc[va_i]
        assert not set(tr["doi_key"]) & set(va["doi_key"]), "DOI leakage!"
        for name, cols in CONFIGS.items():
            Xtr, Xva = encode(tr, va, cols)
            imp = SimpleImputer(strategy="median")
            m = model()
            m.fit(imp.fit_transform(Xtr.values), tr["y"].values)
            yp = m.predict(imp.transform(Xva.values))
            rows.append({"repeat": rep, "fold": k, "config": name,
                         "n_features": len(cols), "r2_log": r2_score(va["y"], yp)})
    print(f"  repeat {rep + 1}/{N_REPEATS} done")

res = pd.DataFrame(rows)

# ---------------------------------------------------------------------------
# 4. Summaries: R² per config, and PAIRED ΔR² vs L3_full
# ---------------------------------------------------------------------------
# R² per repeat = mean over its 5 folds; CI = spread over repeats × folds
per_split = res.pivot_table(index=["repeat", "fold"], columns="config", values="r2_log")

summary = []
for c in per_split.columns:
    v = per_split[c].values
    d = (per_split[c] - per_split["L3_full"]).values
    summary.append({
        "config": c,
        "r2_mean": v.mean(),
        "r2_p2.5": np.percentile(v, 2.5), "r2_p97.5": np.percentile(v, 97.5),
        "delta_vs_L3": d.mean(),
        "delta_p2.5": np.percentile(d, 2.5), "delta_p97.5": np.percentile(d, 97.5),
        "frac_splits_worse_than_L3": (d < 0).mean(),
    })
summary = pd.DataFrame(summary).sort_values("r2_mean", ascending=False)

pd.set_option("display.width", 200)
print(f"\n--- XGBoost, {N_REPEATS}×{N_FOLDS} grouped CV by DOI "
      f"({N_REPEATS * N_FOLDS} paired splits) ---")
print(summary.round(3).to_string(index=False))

print("\nReading guide:")
print("  * EXP_ONLY ≈ L3_full        → material descriptors add nothing (Q1).")
print("  * L3_minus_<g> delta << 0   → group <g> carries the signal (Q2).")
print("  * delta CI crossing 0       → no evidence the group matters.")

# ---------------------------------------------------------------------------
# 5. Plot
# ---------------------------------------------------------------------------
s = summary.set_index("config")
order = list(s.sort_values("r2_mean").index)
fig, ax = plt.subplots(figsize=(8, 0.5 * len(order) + 1.5))
ax.errorbar(s.loc[order, "r2_mean"], range(len(order)),
            xerr=[s.loc[order, "r2_mean"] - s.loc[order, "r2_p2.5"],
                  s.loc[order, "r2_p97.5"] - s.loc[order, "r2_mean"]],
            fmt="o", capsize=3)
ax.axvline(0, color="black", lw=0.8)
ax.axvline(s.loc["L3_full", "r2_mean"], color="grey", ls="--", lw=0.8)
ax.set_yticks(range(len(order))); ax.set_yticklabels(order)
ax.set_xlabel("R² (log scale) — XGBoost, repeated GroupKFold by DOI (2.5–97.5%)")
plt.tight_layout()
plt.savefig("/tmp/nb11_ablation.png", dpi=120)
plt.close()
print("Saved: /tmp/nb11_ablation.png")

# ---------------------------------------------------------------------------
# 6. Write
# ---------------------------------------------------------------------------
spark.createDataFrame(res).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{SCHEMA}.ablation_nb11")
print(f"\n✓ Written: {SCHEMA}.ablation_nb11 ({len(res)} rows)")
print("\n=== NB11 COMPLETE ===")